# TCGA-BRCA Endpoint Crosswalk Review

This notebook is review-only. It loads the latest saved endpoint crosswalk outputs from disk, checks the run-level validation state, and writes review tables for human source audit.


## Load the latest saved endpoint crosswalk run

This section confirms that the stable latest-pointer exists and points to a completed endpoint crosswalk audit run.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')


repo_root = find_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root / '01-data' / 'audit' / 'tcga-brca' / 'variables' / 'tcga_brca_endpoint_crosswalk_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest endpoint crosswalk pointer not found: {latest_pointer_path}. Run the crosswalk script first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
inventory_path = repo_root / latest_pointer['endpoint_candidate_inventory_tsv']
crosswalk_path = repo_root / latest_pointer['endpoint_crosswalk_tsv']
summary_path = repo_root / latest_pointer['endpoint_crosswalk_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']
results_root = repo_root / '09-trials' / '01-tcga-only-source-audited' / '05-results'
results_root.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame([latest_pointer]))


,updated_at_utc,crosswalk_run_id,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,crosswalk_run_directory,endpoint_candidate_inventory_tsv,endpoint_crosswalk_tsv,endpoint_crosswalk_summary_tsv,run_log_json,shortlist_latest_json,core_audit_latest_json,field_count
0,2026-04-12T04:20:36Z,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,01-data/audit/tcga-brca/variables/endpoint_cro...,01-data/audit/tcga-brca/variables/endpoint_cro...,01-data/audit/tcga-brca/variables/endpoint_cro...,01-data/audit/tcga-brca/variables/endpoint_cro...,01-data/audit/tcga-brca/variables/endpoint_cro...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,01-data/audit/tcga-brca/variables/tcga_brca_cl...,13


## Load saved endpoint crosswalk artifacts


In [2]:
inventory_df = pd.read_csv(inventory_path, sep='\t')
crosswalk_df = pd.read_csv(crosswalk_path, sep='\t')
summary_df = pd.read_csv(summary_path, sep='\t')
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

family_order = [
    'survival_status_like',
    'last_contact_like',
    'death_time_like',
    'progression_or_tumor_status_like',
    'new_tumor_event_like',
    'followup_loss_like',
    'other_endpoint_like',
]
role_order = [
    'primary_candidate',
    'secondary_candidate',
    'overlapping_candidate',
    'ambiguous_candidate',
]
candidate_source_order = ['usable_endpoint_candidate', 'escalated_manual_review']

inventory_df['endpoint_signal_family'] = pd.Categorical(
    inventory_df['endpoint_signal_family'], categories=family_order, ordered=True
)
inventory_df['candidate_source'] = pd.Categorical(
    inventory_df['candidate_source'], categories=candidate_source_order, ordered=True
)
inventory_df['manual_review_priority'] = pd.Categorical(
    inventory_df['manual_review_priority'], categories=['high', 'medium', 'low'], ordered=True
)
crosswalk_df['endpoint_signal_family'] = pd.Categorical(
    crosswalk_df['endpoint_signal_family'], categories=family_order, ordered=True
)
crosswalk_df['crosswalk_role'] = pd.Categorical(
    crosswalk_df['crosswalk_role'], categories=role_order, ordered=True
)
crosswalk_df['candidate_source'] = pd.Categorical(
    crosswalk_df['candidate_source'], categories=candidate_source_order, ordered=True
)
crosswalk_df['manual_review_priority'] = pd.Categorical(
    crosswalk_df['manual_review_priority'], categories=['high', 'medium', 'low'], ordered=True
)
summary_df['endpoint_signal_family'] = pd.Categorical(
    summary_df['endpoint_signal_family'], categories=family_order, ordered=True
)

print(f'Endpoint candidate inventory TSV: {inventory_path}')
print(f'Endpoint crosswalk TSV: {crosswalk_path}')
print(f'Endpoint crosswalk summary TSV: {summary_path}')
print(f'Run log: {run_log_path}')
print(f"Crosswalk run ID: {latest_pointer['crosswalk_run_id']}")
print(f"Shortlist run ID: {latest_pointer['shortlist_run_id']}")
print(f"Core audit run ID: {latest_pointer['core_audit_run_id']}")
display(pd.DataFrame([run_log['validation']]))
display(summary_df.sort_values('endpoint_signal_family').reset_index(drop=True))
display(crosswalk_df.head(15))


Endpoint candidate inventory TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\endpoint_crosswalk_runs\20260412T042036Z\endpoint_candidate_inventory.tsv
Endpoint crosswalk TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\endpoint_crosswalk_runs\20260412T042036Z\endpoint_crosswalk.tsv
Endpoint crosswalk summary TSV: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\endpoint_crosswalk_runs\20260412T042036Z\endpoint_crosswalk_summary.tsv
Run log: D:\Projects\brcapath-rx\01-data\audit\tcga-brca\variables\endpoint_crosswalk_runs\20260412T042036Z\run_log.json
Crosswalk run ID: 20260412T042036Z
Shortlist run ID: 20260412T033602Z
Core audit run ID: 20260412T023839Z


,passed,shortlist_latest_pointer_found,core_audit_latest_pointer_found,shortlist_run_log_completed,core_audit_run_log_completed,clinical_biotab_run_log_completed,shortlist_and_core_audit_run_ids_match,patient_and_followup_parsed_tables_found,selected_fields_validated_against_headers_and_schema,at_least_one_usable_endpoint_candidate_found,inventory_row_count_positive,crosswalk_row_count_positive,summary_row_count_positive,inventory_rules_valid,crosswalk_rules_valid,summary_family_counts_match_crosswalk,no_prior_run_overwrite,latest_pointer_written_after_success_only
0,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


,crosswalk_run_id,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,endpoint_signal_family,field_count,tables_present_json,primary_candidates_json,secondary_candidates_json,overlapping_candidates_json,ambiguous_candidates_json,notes_placeholder
0,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,survival_status_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_patient.vital_status""]",[],"[""clinical_follow_up_v4_0.vital_status""]",[],[fill in during endpoint crosswalk review]
1,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,last_contact_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.last_contact_days_to""]",[],"[""clinical_patient.last_contact_days_to""]",[],[fill in during endpoint crosswalk review]
2,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,death_time_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]",[],[],[],"[""clinical_follow_up_v4_0.death_days_to"", ""cli...",[fill in during endpoint crosswalk review]
3,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,progression_or_tumor_status_like,4,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.tumor_status""]",[],"[""clinical_patient.tumor_status""]","[""clinical_patient.days_to_patient_progression...",[fill in during endpoint crosswalk review]
4,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,new_tumor_event_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.new_tumor_event_dx_i...",[],"[""clinical_patient.new_tumor_event_dx_indicator""]",[],[fill in during endpoint crosswalk review]
5,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,followup_loss_like,1,"[""clinical_follow_up_v4_0""]","[""clinical_follow_up_v4_0.followup_lost_to""]",[],[],[],[fill in during endpoint crosswalk review]


,crosswalk_run_id,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,endpoint_signal_family,endpoint_signal_type,endpoint_signal_rule,canonical_signal_key,table_name,...,original_shortlist_rule,selection_rule,missing_like_fraction,non_missing_count,distinct_non_missing_count,example_values_small_sample,crosswalk_role,crosswalk_rule,manual_review_priority,notes_placeholder
0,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,survival_status_like,survival_status_signal,signal_vital_status_name,vital_status,clinical_patient,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.000000,1097,2,"[""Alive"", ""Dead""]",primary_candidate,role_primary_best_usable_completeness,medium,[fill in during endpoint crosswalk review]
1,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,survival_status_like,survival_status_signal,signal_vital_status_name,vital_status,clinical_follow_up_v4_0,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.013966,706,2,"[""Alive"", ""Dead""]",overlapping_candidate,role_overlapping_same_canonical_signal,high,[fill in during endpoint crosswalk review]
2,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,last_contact_like,last_contact_time_signal,signal_last_contact_name,last_contact_days_to,clinical_follow_up_v4_0,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.086592,654,551,"[""10"", ""0"", ""375"", ""396"", ""304""]",primary_candidate,role_primary_best_usable_completeness,medium,[fill in during endpoint crosswalk review]
3,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,last_contact_like,last_contact_time_signal,signal_last_contact_name,last_contact_days_to,clinical_patient,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.094804,993,649,"[""0"", ""10"", ""7"", ""365"", ""30""]",overlapping_candidate,role_overlapping_same_canonical_signal,high,[fill in during endpoint crosswalk review]
4,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,death_time_like,death_time_signal,signal_death_days_to_name,death_days_to,clinical_patient,...,weak_extreme_missingness,escalate_weak_endpoint_like,0.905196,104,101,"[""2965"", ""991"", ""0"", ""1"", ""1009""]",ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,[fill in during endpoint crosswalk review]
5,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,death_time_like,death_time_signal,signal_death_days_to_name,death_days_to,clinical_follow_up_v4_0,...,weak_extreme_missingness,escalate_weak_endpoint_like,0.927374,52,52,"[""0"", ""1034"", ""1048"", ""1072"", ""1093""]",ambiguous_candidate,role_ambiguous_no_usable_primary_or_sparse_signal,high,[fill in during endpoint crosswalk review]
6,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,progression_or_tumor_status_like,tumor_status_signal,signal_tumor_status_name,tumor_status,clinical_follow_up_v4_0,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.054469,677,2,"[""TUMOR FREE"", ""WITH TUMOR""]",primary_candidate,role_primary_best_usable_completeness,medium,[fill in during endpoint crosswalk review]
7,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,progression_or_tumor_status_like,tumor_status_signal,signal_tumor_status_name,tumor_status,clinical_patient,...,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,0.113947,972,2,"[""TUMOR FREE"", ""WITH TUMOR""]",overlapping_candidate,role_overlapping_same_canonical_signal,high,[fill in during endpoint crosswalk review]
8,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,progression_or_tumor_status_like,progression_time_signal,signal_progression_name,days_to_patient_progres

## Save endpoint-candidate inventory review table


In [3]:
inventory_review_columns = [
    'endpoint_signal_family',
    'candidate_source',
    'table_name',
    'field_name',
    'source_position',
    'alternate_column_name',
    'probable_field_group',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'endpoint_signal_type',
    'endpoint_signal_rule',
    'original_shortlist_bucket',
    'original_shortlist_rule',
    'selection_rule',
    'canonical_signal_key',
    'manual_review_priority',
    'example_values_small_sample',
    'notes_placeholder',
]

inventory_review_df = (
    inventory_df.loc[:, inventory_review_columns]
    .sort_values(
        [
            'endpoint_signal_family',
            'candidate_source',
            'missing_like_fraction',
            'non_missing_count',
            'table_name',
            'source_position',
        ],
        ascending=[True, True, True, False, True, True],
    )
    .reset_index(drop=True)
)
inventory_review_path = results_root / '32_endpoint_candidate_inventory.tsv'
inventory_review_df.to_csv(inventory_review_path, sep='\t', index=False)

print(f'Saved: {inventory_review_path}')
display(inventory_review_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\32_endpoint_candidate_inventory.tsv


,endpoint_signal_family,candidate_source,table_name,field_name,source_position,alternate_column_name,probable_field_group,missing_like_fraction,non_missing_count,distinct_non_missing_count,endpoint_signal_type,endpoint_signal_rule,original_shortlist_bucket,original_shortlist_rule,selection_rule,canonical_signal_key,manual_review_priority,example_values_small_sample,notes_placeholder
0,survival_status_like,usable_endpoint_candidate,clinical_patient,vital_status,14,vital_status,follow-up / outcome-like,0.000000,1097,2,survival_status_signal,signal_vital_status_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,vital_status,high,"[""Alive"", ""Dead""]",[fill in during endpoint crosswalk review]
1,survival_status_like,usable_endpoint_candidate,clinical_follow_up_v4_0,vital_status,10,vital_status,follow-up / outcome-like,0.013966,706,2,survival_status_signal,signal_vital_status_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,vital_status,high,"[""Alive"", ""Dead""]",[fill in during endpoint crosswalk review]
2,last_contact_like,usable_endpoint_candidate,clinical_follow_up_v4_0,last_contact_days_to,11,days_to_last_followup,follow-up / outcome-like,0.086592,654,551,last_contact_time_signal,signal_last_contact_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,last_contact_days_to,high,"[""10"", ""0"", ""375"", ""396"", ""304""]",[fill in during endpoint crosswalk review]
3,last_contact_like,usable_endpoint_candidate,clinical_patient,last_contact_days_to,15,days_to_last_followup,follow-up / outcome-like,0.094804,993,649,last_contact_time_signal,signal_last_contact_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,last_contact_days_to,high,"[""0"", ""10"", ""7"", ""365"", ""30""]",[fill in during endpoint crosswalk review]
4,death_time_like,escalated_manual_review,clinical_patient,death_days_to,16,days_to_death,follow-up / outcome-like,0.905196,104,101,death_time_signal,signal_death_days_to_name,weak_or_unusable,weak_extreme_missingness,escalate_weak_endpoint_like,death_days_to,low,"[""2965"", ""991"", ""0"", ""1"", ""1009""]",[fill in during endpoint crosswalk review]
5,death_time_like,escalated_manual_review,clinical_follow_up_v4_0,death_days_to,12,days_to_death,follow-up / outcome-like,0.927374,52,52,death_time_signal,signal_death_days_to_name,weak_or_unusable,weak_extreme_missingness,escalate_weak_endpoint_like,death_days_to,low,"[""0"", ""1034"", ""1048"", ""1072"", ""1093""]",[fill in during endpoint crosswalk review]
6,progression_or_tumor_status_like,usable_endpoint_candidate,clinical_follow_up_v4_0,tumor_status,9,person_neoplasm_cancer_status,follow-up / outcome-like,0.054469,677,2,tumor_status_signal,signal_tumor_status_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,tumor_status,high,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during endpoint crosswalk review]
7,progression_or_tumor_status_like,usable_endpoint_candidate,clinical_patient,tumor_status,13,person_neoplasm_cancer_status,follow-up / outcome-like,0.113947,972,2,tumor_status_signal,signal_tumor_status_name,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,tumor_status,high,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during endpoint crosswalk review]
8,progression_or_tumor_status_like,escalated_manual_review,clinical_patient,days_to_patient_progression_free,97,days_to_patient_progression_free,follow-up / outcome-like,1.000000,0,0,progression_time_signal,signal_progression_name,weak_or_unusable,weak_all_missing,escalate_weak_endpoint_like,days_to_patient_progression_free,low,[],[fill in during endpoint crosswalk review]
9,progression_or_tumor_status_like,escalated_manual_review,clinical_patient,days_to_tumor_progression,98,days_to_tumor_progressio

## Save crosswalk review tables by family and role


In [4]:
crosswalk_review_columns = [
    'endpoint_signal_family',
    'crosswalk_role',
    'candidate_source',
    'table_name',
    'field_name',
    'source_position',
    'alternate_column_name',
    'missing_like_fraction',
    'non_missing_count',
    'distinct_non_missing_count',
    'endpoint_signal_type',
    'endpoint_signal_rule',
    'canonical_signal_key',
    'original_shortlist_bucket',
    'original_shortlist_rule',
    'selection_rule',
    'crosswalk_rule',
    'manual_review_priority',
    'example_values_small_sample',
    'notes_placeholder',
]

crosswalk_by_family_df = (
    crosswalk_df.loc[:, crosswalk_review_columns]
    .sort_values(
        [
            'endpoint_signal_family',
            'crosswalk_role',
            'candidate_source',
            'missing_like_fraction',
            'non_missing_count',
            'table_name',
            'source_position',
        ],
        ascending=[True, True, True, True, False, True, True],
    )
    .reset_index(drop=True)
)
crosswalk_by_family_path = results_root / '33_endpoint_crosswalk_by_family.tsv'
crosswalk_by_family_df.to_csv(crosswalk_by_family_path, sep='\t', index=False)

primary_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'primary_candidate']
    .reset_index(drop=True)
)
primary_candidates_path = results_root / '34_endpoint_crosswalk_primary_candidates.tsv'
primary_candidates_df.to_csv(primary_candidates_path, sep='\t', index=False)

overlapping_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'overlapping_candidate']
    .reset_index(drop=True)
)
overlapping_candidates_path = results_root / '35_endpoint_crosswalk_overlapping_candidates.tsv'
overlapping_candidates_df.to_csv(overlapping_candidates_path, sep='\t', index=False)

ambiguous_candidates_df = (
    crosswalk_by_family_df.loc[crosswalk_by_family_df['crosswalk_role'] == 'ambiguous_candidate']
    .reset_index(drop=True)
)
ambiguous_candidates_path = results_root / '36_endpoint_crosswalk_ambiguous_candidates.tsv'
ambiguous_candidates_df.to_csv(ambiguous_candidates_path, sep='\t', index=False)

print(f'Saved: {crosswalk_by_family_path}')
print(f'Saved: {primary_candidates_path}')
print(f'Saved: {overlapping_candidates_path}')
print(f'Saved: {ambiguous_candidates_path}')
display(summary_df.sort_values('endpoint_signal_family').reset_index(drop=True))
display(primary_candidates_df)
display(overlapping_candidates_df)
display(ambiguous_candidates_df)


Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\33_endpoint_crosswalk_by_family.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\34_endpoint_crosswalk_primary_candidates.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\35_endpoint_crosswalk_overlapping_candidates.tsv
Saved: D:\Projects\brcapath-rx\09-trials\01-tcga-only-source-audited\05-results\36_endpoint_crosswalk_ambiguous_candidates.tsv


,crosswalk_run_id,shortlist_run_id,core_audit_run_id,parse_run_id,source_run_id,endpoint_signal_family,field_count,tables_present_json,primary_candidates_json,secondary_candidates_json,overlapping_candidates_json,ambiguous_candidates_json,notes_placeholder
0,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,survival_status_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_patient.vital_status""]",[],"[""clinical_follow_up_v4_0.vital_status""]",[],[fill in during endpoint crosswalk review]
1,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,last_contact_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.last_contact_days_to""]",[],"[""clinical_patient.last_contact_days_to""]",[],[fill in during endpoint crosswalk review]
2,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,death_time_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]",[],[],[],"[""clinical_follow_up_v4_0.death_days_to"", ""cli...",[fill in during endpoint crosswalk review]
3,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,progression_or_tumor_status_like,4,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.tumor_status""]",[],"[""clinical_patient.tumor_status""]","[""clinical_patient.days_to_patient_progression...",[fill in during endpoint crosswalk review]
4,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,new_tumor_event_like,2,"[""clinical_follow_up_v4_0"", ""clinical_patient""]","[""clinical_follow_up_v4_0.new_tumor_event_dx_i...",[],"[""clinical_patient.new_tumor_event_dx_indicator""]",[],[fill in during endpoint crosswalk review]
5,20260412T042036Z,20260412T033602Z,20260412T023839Z,20260412T010932Z,20260412T000556Z,followup_loss_like,1,"[""clinical_follow_up_v4_0""]","[""clinical_follow_up_v4_0.followup_lost_to""]",[],[],[],[fill in during endpoint crosswalk review]


,endpoint_signal_family,crosswalk_role,candidate_source,table_name,field_name,source_position,alternate_column_name,missing_like_fraction,non_missing_count,distinct_non_missing_count,endpoint_signal_type,endpoint_signal_rule,canonical_signal_key,original_shortlist_bucket,original_shortlist_rule,selection_rule,crosswalk_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,survival_status_like,primary_candidate,usable_endpoint_candidate,clinical_patient,vital_status,14,vital_status,0.000000,1097,2,survival_status_signal,signal_vital_status_name,vital_status,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_primary_best_usable_completeness,medium,"[""Alive"", ""Dead""]",[fill in during endpoint crosswalk review]
1,last_contact_like,primary_candidate,usable_endpoint_candidate,clinical_follow_up_v4_0,last_contact_days_to,11,days_to_last_followup,0.086592,654,551,last_contact_time_signal,signal_last_contact_name,last_contact_days_to,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_primary_best_usable_completeness,medium,"[""10"", ""0"", ""375"", ""396"", ""304""]",[fill in during endpoint crosswalk review]
2,progression_or_tumor_status_like,primary_candidate,usable_endpoint_candidate,clinical_follow_up_v4_0,tumor_status,9,person_neoplasm_cancer_status,0.054469,677,2,tumor_status_signal,signal_tumor_status_name,tumor_status,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_primary_best_usable_completeness,medium,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during endpoint crosswalk review]
3,new_tumor_event_like,primary_candidate,usable_endpoint_candidate,clinical_follow_up_v4_0,new_tumor_event_dx_indicator,13,new_tumor_event_after_initial_treatment,0.096369,647,2,new_tumor_event_signal,signal_new_tumor_event_name,new_tumor_event_dx_indicator,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_primary_best_usable_completeness,medium,"[""NO"", ""YES""]",[fill in during endpoint crosswalk review]
4,followup_loss_like,primary_candidate,usable_endpoint_candidate,clinical_follow_up_v4_0,followup_lost_to,6,lost_follow_up,0.036313,690,2,followup_loss_signal,signal_followup_lost_to_followup_table,followup_lost_to,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_primary_best_usable_completeness,low,"[""NO"", ""YES""]",[fill in during endpoint crosswalk review]


,endpoint_signal_family,crosswalk_role,candidate_source,table_name,field_name,source_position,alternate_column_name,missing_like_fraction,non_missing_count,distinct_non_missing_count,endpoint_signal_type,endpoint_signal_rule,canonical_signal_key,original_shortlist_bucket,original_shortlist_rule,selection_rule,crosswalk_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,survival_status_like,overlapping_candidate,usable_endpoint_candidate,clinical_follow_up_v4_0,vital_status,10,vital_status,0.013966,706,2,survival_status_signal,signal_vital_status_name,vital_status,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_overlapping_same_canonical_signal,high,"[""Alive"", ""Dead""]",[fill in during endpoint crosswalk review]
1,last_contact_like,overlapping_candidate,usable_endpoint_candidate,clinical_patient,last_contact_days_to,15,days_to_last_followup,0.094804,993,649,last_contact_time_signal,signal_last_contact_name,last_contact_days_to,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_overlapping_same_canonical_signal,high,"[""0"", ""10"", ""7"", ""365"", ""30""]",[fill in during endpoint crosswalk review]
2,progression_or_tumor_status_like,overlapping_candidate,usable_endpoint_candidate,clinical_patient,tumor_status,13,person_neoplasm_cancer_status,0.113947,972,2,tumor_status_signal,signal_tumor_status_name,tumor_status,usable_endpoint_candidate,endpoint_candidate_time_or_status_signal,select_usable_endpoint_candidate,role_overlapping_same_canonical_signal,high,"[""TUMOR FREE"", ""WITH TUMOR""]",[fill in during endpoint crosswalk review]
3,new_tumor_event_like,overlapping_candidate,escalated_manual_review,clinical_patient,new_tumor_event_dx_indicator,68,new_tumor_event_after_initial_treatment,0.822242,195,2,new_tumor_event_signal,signal_new_tumor_event_name,new_tumor_event_dx_indicator,unclear_manual_review,manual_default_fallback,escalate_unclear_endpoint_like,role_overlapping_same_canonical_signal,high,"[""NO"", ""YES""]",[fill in during endpoint crosswalk review]


,endpoint_signal_family,crosswalk_role,candidate_source,table_name,field_name,source_position,alternate_column_name,missing_like_fraction,non_missing_count,distinct_non_missing_count,endpoint_signal_type,endpoint_signal_rule,canonical_signal_key,original_shortlist_bucket,original_shortlist_rule,selection_rule,crosswalk_rule,manual_review_priority,example_values_small_sample,notes_placeholder
0,death_time_like,ambiguous_candidate,escalated_manual_review,clinical_patient,death_days_to,16,days_to_death,0.905196,104,101,death_time_signal,signal_death_days_to_name,death_days_to,weak_or_unusable,weak_extreme_missingness,escalate_weak_endpoint_like,role_ambiguous_no_usable_primary_or_sparse_signal,high,"[""2965"", ""991"", ""0"", ""1"", ""1009""]",[fill in during endpoint crosswalk review]
1,death_time_like,ambiguous_candidate,escalated_manual_review,clinical_follow_up_v4_0,death_days_to,12,days_to_death,0.927374,52,52,death_time_signal,signal_death_days_to_name,death_days_to,weak_or_unusable,weak_extreme_missingness,escalate_weak_endpoint_like,role_ambiguous_no_usable_primary_or_sparse_signal,high,"[""0"", ""1034"", ""1048"", ""1072"", ""1093""]",[fill in during endpoint crosswalk review]
2,progression_or_tumor_status_like,ambiguous_candidate,escalated_manual_review,clinical_patient,days_to_patient_progression_free,97,days_to_patient_progression_free,1.000000,0,0,progression_time_signal,signal_progression_name,days_to_patient_progression_free,weak_or_unusable,weak_all_missing,escalate_weak_endpoint_like,role_ambiguous_no_usable_primary_or_sparse_signal,high,[],[fill in during endpoint crosswalk review]
3,progression_or_tumor_status_like,ambiguous_candidate,escalated_manual_review,clinical_patient,days_to_tumor_progression,98,days_to_tumor_progression,1.000000,0,0,progression_time_signal,signal_progression_name,days_to_tumor_progression,weak_or_unusable,weak_all_missing,escalate_weak_endpoint_like,role_ambiguous_no_usable_primary_or_sparse_signal,high,[],[fill in during endpoint crosswalk review]
